# E07 — Standard RAG (A2) — Analysis

**Research question**: Does `retrieval_v1` improve `qwen2.5:7b-instruct` classification and
evidence-grounding relative to full-context input (E05) when evaluated on the same frozen
cases?

**A2 architecture**: requirement/hypothesis -> `retrieval_v1` (BM25 -> clause_256 -> top-20 ->
rerank -> top-5, unmodified) -> `classification_prompt_v1` (unchanged) -> "Retrieved NDA
excerpts:" wrapper -> `qwen2.5:7b-instruct-ctx16k` (same tag as E05) -> compact label/evidence
schema -> deterministic parser -> evidence validator. No agent, no iteration, no query
rewriting.

**Frozen components** (matched to E05): `TRAIN_ARCH_v1` manifest (same 150 cases, same order),
model tag, temperature 0, timeout 60s, prompt, output schema, parser, evidence validator.
**Only the input-construction architecture differs.**

Local-only, $0, zero hosted calls.

In [1]:
import csv
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "E07_standard_rag" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
E07 = REPO_ROOT / "experiments/E07_standard_rag"
E05 = REPO_ROOT / "experiments/E05_full_context"
RESULTS = E07 / "results"

manifest = json.load(open(E05 / "TRAIN_ARCH_v1.json"))
summary = json.load(open(RESULTS / "run_E07_A2_train.json"))
wall = json.load(open(RESULTS / "run_E07_A2_train_wall_seconds.json"))
paired = json.load(open(RESULTS / "e05_vs_e07_paired_comparison.json"))
e05_summary = json.load(open(E05 / "results/run_E05_A1_train.json"))
print("n_cases:", summary["n_cases"], "  total wall time (min):", round(wall["total_wall_seconds"]/60, 1))

n_cases: 150   total wall time (min): 19.2


## 4. TRAIN_ARCH_v1 verification (identical manifest to E05)

In [2]:
from collections import Counter
balance = Counter(c["gold_label"] for c in manifest["cases"])
assert balance == Counter({"Entailment": 50, "Contradiction": 50, "NotMentioned": 50})
assert manifest["seed"] == 700 and manifest["total_unique_documents"] == 78
print("Verified: 150 cases, 50/50/50, seed=700, 78 unique documents -- identical to E05.")

Verified: 150 cases, 50/50/50, seed=700, 78 unique documents -- identical to E05.


## 5. Retrieval coverage on TRAIN_ARCH_v1 (characterization, retrieval_v1 unmodified)

In [3]:
print("Evidence Recall@5 overall: 90.9%  (measured in Stage A, see config.yaml)")
print("Entailment: 85.4%   Contradiction: 96.3%   Miss count: 6/100")

Evidence Recall@5 overall: 90.9%  (measured in Stage A, see config.yaml)
Entailment: 85.4%   Contradiction: 96.3%   Miss count: 6/100


## 6. Token reduction vs. E05

In [4]:
tr = paired["token_reduction"]
print(f"Mean reduction:   {tr['mean_pct']:.1f}%")
print(f"Median reduction: {tr['median_pct']:.1f}%")
print(f"P90 reduction:    {tr['p90_pct']:.1f}%")
print()
print("Absolute (E05 -> E07):")
print(f"  mean:   {e05_summary['input_tokens']['mean']:.0f} -> {summary['input_tokens']['mean']:.0f}")
print(f"  median: {e05_summary['input_tokens']['median']:.0f} -> {summary['input_tokens']['median']:.0f}")
print(f"  p90:    {e05_summary['input_tokens']['p90']:.0f} -> {summary['input_tokens']['p90']:.0f}")

Mean reduction:   47.8%
Median reduction: 41.8%
P90 reduction:    66.0%

Absolute (E05 -> E07):
  mean:   2253 -> 1177
  median: 2046 -> 1192
  p90:    3945 -> 1341


## 7-8. Classification metrics and confusion matrix

In [5]:
cls = summary["classification"]
print(f"Accuracy:  {cls['accuracy']:.4f}")
print(f"Macro-F1:  {cls['macro_f1']:.4f}")
for label, r in cls["per_class_recall"].items():
    print(f"  {label:>13}: {r:.4f}")
cm = cls["confusion_matrix"]
print(f"\nConfusion matrix (rows=gold, cols=predicted, order={cm['labels']}):")
for label, row in zip(cm["labels"], cm["matrix"]):
    print(f"  {label:>13}: {row}")

Accuracy:  0.4333
Macro-F1:  0.4309
     Entailment: 0.3400
  Contradiction: 0.4200
   NotMentioned: 0.5400

Confusion matrix (rows=gold, cols=predicted, order=['Entailment', 'Contradiction', 'NotMentioned']):
     Entailment: [17, 17, 16]
  Contradiction: [3, 21, 26]
   NotMentioned: [12, 11, 27]


## 9. Contradiction Recall

In [6]:
lo, hi = cls["contradiction_recall_ci95"]
print(f"Contradiction Recall: {cls['contradiction_recall']:.4f} (n={cls['contradiction_n']}, 95% CI [{lo:.4f}, {hi:.4f}])")

Contradiction Recall: 0.4200 (n=50, 95% CI [0.2937, 0.5577])


## 10. Structured-output metrics — never collapsed

In [7]:
so = summary["structured_output"]
print(f"Strict:    {so['strict_n']}/{summary['n_cases']} ({so['strict_parse_validity_pct']:.1f}%)")
print(f"Recovered: {so['recovered_n']}/{summary['n_cases']}")
print(f"Invalid:   {so['invalid_n']}/{summary['n_cases']}")
print(f"Usable:    {so['usable_parse_validity_pct']:.1f}%")
print(f"Retries: {so['total_retries']}  Model errors: {so['model_errors']}  Timeouts: {so['timeouts']}")

Strict:    150/150 (100.0%)
Recovered: 0/150
Invalid:   0/150
Usable:    100.0%
Retries: 0  Model errors: 0  Timeouts: 0


## 11. Evidence metrics

In [8]:
ev = summary["evidence"]
print(f"Evidence Recall:    {ev['evidence_recall']:.4f}")
print(f"Evidence Precision: {ev['evidence_precision']:.4f}")
print(f"Correct-label-but-bad-evidence: {ev['correct_label_bad_evidence_count']}")
print(f"Wrong-label-but-valid-evidence: {ev['wrong_label_valid_evidence_count']}")
print(f"Paraphrased evidence: {ev['paraphrased_evidence_count']}")

Evidence Recall:    0.2900
Evidence Precision: 0.3580
Correct-label-but-bad-evidence: 14
Wrong-label-but-valid-evidence: 54
Paraphrased evidence: 32


## 12. Joint label+evidence success

In [9]:
joint = summary["joint"]
print(f"Overall: {joint['overall']:.4f}")
for label, rate in joint["by_class"].items():
    print(f"  {label:>13}: {rate:.4f}")

Overall: 0.3333
     Entailment: 0.1600
  Contradiction: 0.3000
   NotMentioned: 0.5400


## 13. Retrieval-limited vs. reasoning-limited decomposition

Evaluator-side only -- gold-evidence-presence was never shown to the model.

In [10]:
counts = summary["retrieval_aware_failure_counts"]
for fam, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"{n:>4}  {fam}")
print("\nContradiction-specific:")
c_counts = summary["retrieval_aware_failure_counts_contradiction"]
for fam, n in sorted(c_counts.items(), key=lambda x: -x[1]):
    print(f"{n:>4}  {fam}")

  79  B_reasoning_limited
  51  n/a (correct)
  14  C_evidence_selection_failure
   6  A_retrieval_limited

Contradiction-specific:
  28  B_reasoning_limited
  14  n/a (correct)
   7  C_evidence_selection_failure
   1  A_retrieval_limited


## 14. Latency — retrieval, generation, end-to-end (retrieval overhead not hidden)

In [11]:
print("Retrieval latency (ms):", summary["retrieval_latency_ms"])
print("Generation latency (ms):", summary["generation_latency_ms"])
print("End-to-end latency (ms):", summary["end_to_end_latency_ms"])
print()
print("E05 generation latency (ms) for comparison:", e05_summary["latency_ms"])

Retrieval latency (ms): {'mean': 416.29653693409637, 'median': 405.52210400346667, 'p90': 645.3235000371933, 'max': 2999.351041042246}
Generation latency (ms): {'mean': 7266.066341951179, 'median': 6106.188000005204, 'p90': 11730.843666009605, 'max': 19851.623042020947}
End-to-end latency (ms): {'mean': 7682.362878885276, 'median': 6516.4434795151465, 'p90': 12370.24737498723, 'max': 20327.212750911713}

E05 generation latency (ms) for comparison: {'mean': 12763.680564106908, 'median': 10174.425395554863, 'p90': 22214.195166015998, 'max': 84038.91895792913}


## 15. E05 vs. E07 matched metric table

In [12]:
import pandas as pd
rows = []
for metric, vals in paired["metric_table"].items():
    rows.append({"metric": metric, "E05": vals["E05"], "E07": vals["E07"], "delta_E07_minus_E05": vals["delta_E07_minus_E05"]})
pd.DataFrame(rows).set_index("metric")

,E05,E07,delta_E07_minus_E05
metric,,,
accuracy,0.400000,0.433333,0.033333
macro_f1,0.397437,0.430886,0.033449
entailment_recall,0.480000,0.340000,-0.140000
contradiction_recall,0.280000,0.420000,0.140000
notmentioned_recall,0.440000,0.540000,0.100000
joint_overall,0.280000,0.333333,0.053333
strict_parse_rate,0.940000,1.000000,0.060000
usable_parse_rate,1.000000,1.000000,0.000000
evidence_recall,0.250000,0.290000,0.040000


## 16. Case-level transitions

In [13]:
print("Overall:", paired["transitions_overall"])
print("Contradiction only:", paired["transitions_contradiction"])

Overall: {'e05_wrong_e07_correct': 24, 'e05_correct_e07_wrong': 19, 'both_correct': 41, 'both_wrong': 66}
Contradiction only: {'e05_wrong_e07_correct': 11, 'e05_correct_e07_wrong': 4, 'both_correct': 10, 'both_wrong': 25}


## 17. McNemar's exact test

In [14]:
print("Overall:", paired["mcnemar_overall"])
print("Contradiction subset:", paired["mcnemar_contradiction"])
print()
print("Interpretation: a p-value alone does not determine the architecture decision -- see the")
print("effect size (bootstrap deltas, next cell) interpreted alongside it.")

Overall: {'b_e05_only_correct': 19, 'c_e07_only_correct': 24, 'n_discordant': 43, 'p_value': 0.542384011856484, 'significant_at_0.05': False}
Contradiction subset: {'b_e05_only_correct': 4, 'c_e07_only_correct': 11, 'n_discordant': 15, 'p_value': 0.11846923828124999, 'significant_at_0.05': False}

Interpretation: a p-value alone does not determine the architecture decision -- see the
effect size (bootstrap deltas, next cell) interpreted alongside it.


## 18. Paired bootstrap (10,000 resamples)

In [15]:
print("Accuracy delta (E07-E05):", paired["bootstrap_accuracy_delta_E07_minus_E05"])
print("Contradiction recall delta (E07-E05):", paired["bootstrap_contradiction_recall_delta_E07_minus_E05"])

Accuracy delta (E07-E05): {'point_estimate': 0.033333333333333326, 'ci95_low': -0.053333333333333344, 'ci95_high': 0.12}
Contradiction recall delta (E07-E05): {'point_estimate': 0.13999999999999996, 'ci95_low': 0.0, 'ci95_high': 0.28}


## 19. Representative improvements / regressions

In [16]:
e05_cases = {json.loads(l)["case_id"]: json.loads(l) for l in open(E05 / "results/run_E05_A1_train_cases.jsonl")}
e07_cases = {json.loads(l)["case_id"]: json.loads(l) for l in open(RESULTS / "run_E07_A2_train_cases.jsonl")}
rag_rows = list(csv.DictReader(open(RESULTS / "rag_failure_analysis.csv")))

print("--- E05 wrong -> E07 correct (retrieval helped) ---")
n = 0
for cid, e05c in e05_cases.items():
    e07c = e07_cases[cid]
    if e05c["predicted_label"] != e05c["gold_label"] and e07c["predicted_label"] == e07c["gold_label"]:
        print(f"{cid}  gold={e05c['gold_label']}  E05_pred={e05c['predicted_label']}  E07_pred={e07c['predicted_label']}")
        n += 1
        if n >= 3:
            break

print("\n--- E05 correct -> E07 wrong (retrieval hurt) ---")
n = 0
for cid, e05c in e05_cases.items():
    e07c = e07_cases[cid]
    if e05c["predicted_label"] == e05c["gold_label"] and e07c["predicted_label"] != e07c["gold_label"]:
        print(f"{cid}  gold={e05c['gold_label']}  E05_pred={e05c['predicted_label']}  E07_pred={e07c['predicted_label']}")
        n += 1
        if n >= 3:
            break

--- E05 wrong -> E07 correct (retrieval helped) ---
train::88::nda-2  gold=Contradiction  E05_pred=NotMentioned  E07_pred=Contradiction
train::106::nda-2  gold=Contradiction  E05_pred=NotMentioned  E07_pred=Contradiction
train::178::nda-20  gold=Contradiction  E05_pred=NotMentioned  E07_pred=Contradiction

--- E05 correct -> E07 wrong (retrieval hurt) ---
train::92::nda-17  gold=Contradiction  E05_pred=Contradiction  E07_pred=NotMentioned
train::92::nda-20  gold=Contradiction  E05_pred=Contradiction  E07_pred=NotMentioned
train::141::nda-7  gold=Contradiction  E05_pred=Contradiction  E07_pred=NotMentioned


## 20. Failure analysis (families, not forced)

In [17]:
ff = summary["failure_family_counts"]
for fam, n in sorted(ff.items(), key=lambda x: -x[1]):
    print(f"{n:>4}  {fam}")

  37  NotMentioned_overprediction (evidence was present)
  22  reasoning_failure_evidence_present
  18  evidence_paraphrasing
   5  retrieval_miss (NotMentioned overprediction, evidence absent)
   3  exception_carveout_candidate


## 21. Final A2 characterization

`A2_standard_rag_v1` — frozen, reproducible architecture configuration (not a claim of
production-readiness): `retrieval_v1` (unmodified) → `classification_prompt_v1` (unchanged) →
`qwen2.5:7b-instruct-ctx16k` → compact schema → deterministic parser → evidence validator,
`TRAIN_ARCH_v1` manifest. Full metrics above and in `results/run_E07_A2_train.json`. Per
instruction, no change was made to retrieval_v1, the prompt, the parser, the model, or the
timeout in response to these results — all findings (whichever direction they point) are
recorded as real results of this matched comparison, for E12's architecture decision to weigh
alongside E04 (A0)/E05 (A1) and the future agent-based architecture.